# Major Montana landowners FINAL analysis



In [12]:
# Library import and config
import json
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_rows', 100) # Avoids truncation of long data table results
pd.set_option('display.max_columns', 100) # Avoids truncation of wide data table results

In [13]:
# Read in data (slow)
# Parcel data from **March 9** download -- may need to update file path
parcels = gpd.read_file('./raw/MontanaCadastral_SHP_3-9-2026/Montana_Cadastral/OWNERPARCEL.shp').to_crs(epsg=4326)



In [14]:
# Add in ownership groupings

with open('./owner-name-grouping.json', 'r') as file:
    OWNER_GROUPS_RAW = json.load(file)

cleaner = {}
for group in OWNER_GROUPS_RAW:
    ownerCleaned = group['owner']
    for ownerAlternate in group['alternatesOwnershipNames']:
        cleaner[ownerAlternate] = ownerCleaned
parcels['OwnerName_Grouped'] = parcels['OwnerName'].replace(cleaner)



In [15]:
# Group parcels by owner name, count the number in each group and sum by acreage in each group
# Running more than top 10 here because most of the biggest are public entities, which we need to filter out
top_100_landowners = parcels.groupby(['OwnerName_Grouped']).agg({
    'PARCELID': 'count',
    'TotalAcres': 'sum',
    'OwnerName': 'unique',
}).sort_values('TotalAcres', ascending=False).head(100)

top_100_landowners.head(10) # Show top 10 to check output

,PARCELID,TotalAcres,OwnerName
OwnerName_Grouped,,,
USDA FOREST SERVICE,16672,8775511.897,[USDA FOREST SERVICE]
USDI BUREAU OF LAND MANAGEMENT,18102,5604489.400,[USDI BUREAU OF LAND MANAGEMENT]
STATE OF MONTANA,12223,4265228.436,[STATE OF MONTANA]
UNITED STATES OF AMERICA,5580,1881103.943,[UNITED STATES OF AMERICA]
USA,3985,1828458.927,[USA]
USA IN TRUST FOR CROW TRIBE,3848,1303624.520,[USA IN TRUST FOR CROW TRIBE]
USA IN TRUST,5279,1139955.995,[USA IN TRUST]
USA - FOREST SERVICE,2161,1115836.824,[USA - FOREST SERVICE]
BUREAU OF LAND MANAGEMENT,2735,732067.380,[BUREAU OF LAND MANAGEMENT]


In [16]:
# Filter out manually identified public landowners

with open('./known-public-landowners.json', 'r') as file:
    PUBLIC_LANDOWNERS = json.load(file)

top_100_private_only = top_100_landowners[~top_100_landowners.index.isin(PUBLIC_LANDOWNERS)]

# Add rank for display purposes
ranking = top_100_private_only.copy().reset_index()
ranking['Rank'] = ranking.index+1
ranking.rename(columns={
    'PARCELID': 'NumParcels',
    'OwnerName_Grouped': 'Owner',
    'OwnerName': 'IncludedOwnerNames',
    }, inplace=True)
ranking.head(30)[['Rank','Owner','NumParcels','TotalAcres',]] # show top 20

,Rank,Owner,NumParcels,TotalAcres
0,1,GALT RANCH,820,346571.264
1,2,KROENKE GROUP,889,331063.014
2,3,FARMLAND RESERVE INC,670,324889.022
3,4,GREEN DIAMOND RESOURCE COMPANY,644,291802.938
4,5,WILKS BROTHERS,705,275634.288
5,6,SUNLIGHT RANCH COMPANY,733,240805.725
6,7,COFFEE-NEFSY,503,238831.067
7,8,NATURE CONSERVANCY,454,157605.619
8,9,AMERICAN PRAIRIE,606,145969.964
9,10,GREAT NORTHERN PROPERTIES,294,144144.127


In [17]:
# Write top 20 list to outputs file as JSON
ranking.head(20).to_json('./outputs/final-top-20-list.json', orient='records', indent=4, index=False)

In [23]:
# More human-readable listing with acreages for each underlying owner name
# Manually copy and paste to outputs/

string = ''

for idx, row in ranking.head(20).iterrows():
    string += ('#' + str(idx+1) + ' / ' + row['Owner'])
    string += (f'\n{row['TotalAcres']:,.0f} acres - {str(row['NumParcels'])} parcels')
    string += ('\n  Includes:')
    for subOwner in row['IncludedOwnerNames']:
        subOwnerParcels = parcels[parcels['OwnerName'] == subOwner]
        string +=(f'\n    {subOwner} --> {subOwnerParcels['TotalAcres'].sum():,.0f} acres/ {len(subOwnerParcels)} parcels')
    string += ('\n\n')

print(string)
with open("./outputs/final-top-20-list.txt", "w") as f:
    f.write(string)

#1 / GALT RANCH
346,571 acres - 820 parcels
  Includes:
    SUN COULEE LLC --> 4,508 acres/ 20 parcels
    71 RANCH LP --> 249,141 acres/ 616 parcels
    GALT RANCH LP --> 92,922 acres/ 182 parcels
    GALT ERROL T --> 1 acres/ 1 parcels
    GALT ERROL T & SHARRIE K --> 0 acres/ 1 parcels

#2 / KROENKE GROUP
331,063 acres - 889 parcels
  Includes:
    PV RANCH COMPANY LLC --> 116,672 acres/ 212 parcels
    PV RANCH CO LLC --> 16,312 acres/ 32 parcels
    FORT PEASE COMM PASTURE --> 20,847 acres/ 41 parcels
    PV RANCH HOLDINGS LLC --> 9,706 acres/ 23 parcels
    BROKEN O LAND & LIVESTOCK LLC --> 119,111 acres/ 454 parcels
    PV ANTELOPE CREEK HOLDINGS LLC --> 29,487 acres/ 86 parcels
    KROENKE LAND & LIVESTOCK LLC --> 12,485 acres/ 25 parcels
    KROENKE LAND & LIVESTOCK CO LLC --> 3,332 acres/ 10 parcels
    KROENKE LAND & LIVESTOCK --> 651 acres/ 2 parcels
    KROENKE LAND & LIVESTOCK COMPANY LLC --> 539 acres/ 1 parcels
    FORT PEASE COMMUNITY PASTURE INC --> 1,280 acres/ 2 par

In [19]:
# # Write parcels to separate file to allow analysis using MTFP-assigned groupings in explore.ipyb
parcels.to_file('./cleaned/parcels-with-mtfp-groupings.shp')

/var/folders/f6/r93y9f4n6w1f58bmth5bzr180000gn/T/ipykernel_6208/2188858717.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  parcels.to_file('./cleaned/parcels-with-mtfp-groupings.shp')
/Users/edietrich/.pyenv/versions/3.13.0/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'OwnerName_Grouped' to 'OwnerName_'
  ogr_write(
/Users/edietrich/.pyenv/versions/3.13.0/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 112736384.070999995 of field Shape_Area of feature 108015 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
/Users/edietrich/.pyenv/versions/3.13.0/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 357951429.420000017 of field Shape_Area of feature 391258 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
/Users/edietrich/.pyenv/versi

In [20]:
# Export geojson data for the parcels owned by each of the top 20 landowners 
majorColumns = ['CountyName','OwnerName_Grouped','OwnerName','OwnerAddre','OwnerAdd_1','OwnerAdd_2','OwnerCity','OwnerState','PropType','TotalAcres','geometry']
for idx, row in ranking.head(20).iterrows():
    subOwnerParcels = parcels[parcels['OwnerName_Grouped'] == row['Owner']][majorColumns]
    path = f'./geodata-outputs/{str(idx+1)}-{row['Owner'].replace(' ','-').replace('&-','')}.geojson'
    subOwnerParcels.to_file(path, driver="GeoJSON")

In [ ]:
# Export to combined CSV/GeoJSON files
parcelsInTop10 = parcels[parcels['OwnerName_Grouped'].isin(list(ranking.head(10)['Owner']))]
top10OwnerRank = ranking.head(10)[['Owner', 'Rank']]
mergedTop10Parcels = parcelsInTop10.merge(top10OwnerRank, left_on='OwnerName_Grouped', right_on='Owner', how='left')
mergedTop10Parcels.drop(columns='Owner',inplace=True)
mergedTop10Parcels.to_file('./geodata-outputs/top-10-combined.geojson', driver='GeoJSON')[majorColumns]
mergedTop10Parcels.to_csv('./outputs/top-10-itemized.csv', index=False)
                           